In [1]:
import os
import requests

API_BASE = "http://35.184.189.38/api"
API_KEY  = os.environ["HMDB_API_KEY"]  # put your real key in an env var

def with_key(params=None):
    params = dict(params or {})
    params["api-key"] = API_KEY
    return params

# 1) GET a single metabolite by HMDB ID
r = requests.get(f"{API_BASE}/hmdb/metabolites/HMDB0000001/", params=with_key())
r.raise_for_status()
met = r.json()
print(met["name"], met.get("chemical_formula"))

# 2) POST pathways for multiple HMDB IDs
payload = {"hmdb_id": ["HMDB0000001", "HMDB0000002"]}
r = requests.post(f"{API_BASE}/hmdb/metabolites/pathways/", params=with_key(), json=payload)
r.raise_for_status()
pathways = r.json()
print(pathways)

1-Methylhistidine None
{'found': [{'hmdb_id': 'HMDB0000001', 'pathways': [{'smpdb_id': 'SMP0000044', 'name': 'Histidine Metabolism', 'description': 'Histidine, an amino acid, plays an important role in the creation of proteins. It is unique as an amino acid as it is needed for nucleotide formation. The biosynthesis of histidine in adults begins with the condensation of ATP and PRPP (phosphoribosyl pyrophosphate) to form n-5-phosphoribosyl 1-pyrophosphate (phosphoribosyl-ATP). It is also worth noting that PRPP is the beginning compound for purine and pyrimidine creation. Subsequent histidine biosynthetic steps (from phosphoribosyl-ATP onwards) are likely to occur in the intestinal microflora. Elimination of the phosphate and the opening of the ring in phosphoribosyl-ATP forms phosphoribosyl-forminino-5-aminoimidazole-4-carboxamide ribonucleotide(phosphoribosyl-forminino-AICAR-phosphate). This is subsequently converted to 5-phosphoribulosyl-forminino-5-aminoimidazole-4-carboxamide ribonu

In [3]:
import os
from hmdb_client import HMDBClient

# Prefer header token, exactly like your curl:
# export HMDB_TOKEN="REMOVED_CREDENTIAL"
client = HMDBClient(token=os.environ["HMDB_API_KEY"])

# Matches your curl JSON body; provide only the fields you care about:
res = client.post(
    "/hmdb/metabolites/search/",
    json_body={"name": "1-methylhistidine"},
    params=None,
)
print(res)

# Other endpoints still work (GET by ID, POST pathways). For those, you can
# also pass an API key instead of a token if that’s what those routes expect:
# client = HMDBClient(api_key=os.environ["HMDB_API_KEY"])


[{'hmdb_id': 'HMDB0000001', 'status': 'Detected and Quantified', 'created_at': '2005-11-16T15:48:42Z', 'updated_at': '2023-02-21T17:14:26Z', 'name': '1-Methylhistidine', 'description': "1-Methylhistidine, also known as 1-MHis, 1MH, tau-methylhistidine or tele-methylhistidine, belongs to the class of organic compounds known as histidine and derivatives. 1MH is also classified as a methylamino acid. Methylamino acids are primarily proteogenic amino acids (found in proteins) which have been methylated (in situ) on their side chains by various methyltransferase enzymes. Histidine can be methylated at either the N1 or N3 position of its imidazole ring, yielding the isomers 1-methylhistidine (1MH; also referred to as tau-methylhistidine, according to IUPAC) or 3-methylhistidine (3MH; pi-methylhistidine, according to IUPAC), respectively. There is considerable confusion with regard to the nomenclature of the methylated nitrogen atoms on the imidazole ring of histidine in histidine-containing 

In [13]:
from __future__ import annotations
from typing import Optional, List, Dict, Any, Type
from pydantic import PrivateAttr
from pydantic import BaseModel, Field
import json
import logging
import os

# Optional deps
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors
    RDKIT_AVAILABLE = True
except Exception:
    Chem = None
    Descriptors = None
    rdMolDescriptors = None
    RDKIT_AVAILABLE = False

try:
    from langchain.tools import BaseTool
except Exception:
    class BaseTool:
        pass

# You already have this in your env / client module
try:
    from hmdb_client import HMDBClient
except Exception as _e:
    HMDBClient = None

log = logging.getLogger("hmdb_agent")
if not log.handlers:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")


def safe_tool_wrapper(func):
    def wrapper(*args, **kwargs):
        try:
            out = func(*args, **kwargs)
            if out is None or (isinstance(out, str) and out.strip() == ""):
                return "No information available for this query."
            return out
        except Exception as e:
            cls_name = args[0].__class__.__name__ if args and hasattr(args[0], "__class__") else "Tool"
            log.exception(f"{cls_name}.{func.__name__} failed: {e}")
            return json.dumps({"error": True, "message": f"{cls_name}.{func.__name__} failed: {str(e)}"})
    return wrapper


# ---------------------------
# Input schema
# ---------------------------
class HMDBInput(BaseModel):
    name: Optional[str] = Field(None, description="Metabolite name (e.g., '1-methylhistidine').")
    hmdb_id: Optional[str] = Field(None, description="HMDB accession ID (e.g., 'HMDB0000001').")
    formula: Optional[str] = Field(None, description="Molecular formula.")
    smiles: Optional[str] = Field(None, description="SMILES.")
    inchikey: Optional[str] = Field(None, description="InChIKey.")
    return_format: str = Field("json", description="One of: json|markdown|both")


# ---------------------------
# Tool
# ---------------------------
class HMDBDatabaseTool(BaseTool):
    name: str = "hmdb_database"
    description: str = (
        "Query HMDB’s metabolites search API and normalize results to a compact, agent-friendly schema. "
        "Enrich with RDKit if available. Returns JSON by default; can also render markdown."
    )
    args_schema: Type[BaseModel] = HMDBInput

    _client: Any = PrivateAttr(default=None)

    token_env: str = "HMDB_API_KEY"

    def __init__(self, token_env: str = "HMDB_API_KEY", **data):
        super().__init__(token_env=token_env, **data)

        token = os.environ.get(token_env) or os.environ.get("HMDB_TOKEN")
        if not token:
            raise RuntimeError(f"Missing HMDB token in env vars '{token_env}' or 'HMDB_TOKEN'.")

        if HMDBClient is None:
            raise RuntimeError("hmdb_client is not installed/importable.")

        self._client = HMDBClient(token=token)

    # LangChain sync entry
    @safe_tool_wrapper
    def _run(
        self,
        name: Optional[str] = None,
        hmdb_id: Optional[str] = None,
        formula: Optional[str] = None,
        smiles: Optional[str] = None,
        inchikey: Optional[str] = None,
        return_format: str = "json",
    ) -> str:
        payload = self._core_query(
            name=name,
            hmdb_id=hmdb_id,
            formula=formula,
            smiles=smiles,
            inchikey=inchikey,
        )
        return self._format_payload(payload, return_format)

    # Async stub if your agent runtime expects it
    async def _arun(self, **kwargs) -> str:
        return self._run(**kwargs)

    # ---------------------------
    # Core query + normalize
    # ---------------------------
    def _core_query(
        self,
        name: Optional[str],
        hmdb_id: Optional[str],
        formula: Optional[str],
        smiles: Optional[str],
        inchikey: Optional[str],
    ) -> Dict[str, Any]:
        if not any([name, hmdb_id, formula, smiles, inchikey]):
            return {"error": True, "message": "Provide at least one of: name, hmdb_id, formula, smiles, inchikey."}

        body = {
            "name": name,
            "hmdb_id": hmdb_id,
            "formula": formula,
            "smiles": smiles,
            "inchi_key": inchikey,
        }
        body = {k: v for k, v in body.items() if v}

        try:
            # ✅ use the private client
            res = self._client.post("/hmdb/metabolites/search/", json_body=body, params=None)
            records: List[Dict[str, Any]] = res if isinstance(res, list) else []
        except Exception as e:
            return {"error": True, "message": f"HMDB request failed: {e}"}

        if not records:
            return {"error": True, "message": "No HMDB records found for this query."}

        normed = [self._normalize_hmdb_record(r) for r in records]
        primary = normed[0]
        candidates = normed[:10]

        syns = []
        for r in records:
            for s in (r.get("synonyms") or []):
                n = (s.get("name") or "").strip()
                if n and n not in syns:
                    syns.append(n)

        payload = {
            "error": False,
            "primary": primary["core"],
            "candidates": [c["core"] for c in candidates],
            "synonyms": syns[:50],
            "identifiers": primary["identifiers"],
            "taxonomy": primary["taxonomy"],
            "physchem": primary["physchem"],
            "retention_indices": primary["retention_indices"],
            "evidence": primary["evidence"],
            "references": primary["references"],
            "notes": primary["notes"],
        }
        return payload

    # ---------------------------
    # Normalizers
    # ---------------------------
    def _normalize_hmdb_record(self, r: Dict[str, Any]) -> Dict[str, Any]:
        # Core identity
        hmdb_id = r.get("hmdb_id") or ""
        name = r.get("name") or ""
        status = r.get("status") or ""
        formula = r.get("moldb_formula") or ""
        smiles = r.get("moldb_smiles") or ""
        inchi = r.get("moldb_inchi") or ""
        inchikey = r.get("moldb_inchikey") or ""
        cas = r.get("cas") or ""
        mono = _safe_float(r.get("moldb_mono_mass"))
        avg = _safe_float(r.get("moldb_average_mass"))

        # RDKit enrich if possible from SMILES
        rd_exact = None
        rd_formula = None
        if smiles and RDKIT_AVAILABLE:
            try:
                mol = Chem.MolFromSmiles(smiles)
                if mol:
                    if rdMolDescriptors:
                        rd_exact = round(rdMolDescriptors.CalcExactMolWt(mol), 6)
                        rd_formula = rdMolDescriptors.CalcMolFormula(mol)
            except Exception:
                pass

        # Identifiers/external links
        xref = r.get("external_links") or {}
        identifiers = {
            "hmdb_id": hmdb_id,
            "inchikey": inchikey,
            "inchi": inchi,
            "cas": cas,
            "pubchem_cid": (xref.get("pubchem_compound") or "") if isinstance(xref, dict) else "",
            "chebi_id": (xref.get("chebi_id") or "") if isinstance(xref, dict) else "",
            "kegg_id": (xref.get("kegg_id") or "") if isinstance(xref, dict) else "",
            "drugbank_id": (xref.get("drugbank_id") or "") if isinstance(xref, dict) else "",
            "chemspider_id": (xref.get("chemspider_id") or "") if isinstance(xref, dict) else "",
            "metlin_id": (xref.get("metlin_id") or "") if isinstance(xref, dict) else "",
            "wikipedia": (xref.get("wikipedia_id") or "") if isinstance(xref, dict) else "",
            "markerdb_id": (xref.get("markerdb_id") or "") if isinstance(xref, dict) else "",
            "foodb_id": (xref.get("foodb_id") or "") if isinstance(xref, dict) else "",
        }

        # Taxonomy (compact but useful)
        tax = r.get("chemical_taxonomy") or {}
        taxonomy = {
            "kingdom": tax.get("kingdom") or "",
            "superclass": tax.get("superclass") or "",
            "class": tax.get("class") or "",
            "subclass": tax.get("subclass") or "",
            "direct_parent": tax.get("direct_parent") or "",
            "molecular_framework": tax.get("molecular_framework") or "",
            "alternative_parents": tax.get("alternative_parents") or [],
            "external_descriptors": tax.get("external_descriptors") or [],
        }

        # Physchem
        physchem = {
            "logp_alogps": _safe_float(r.get("moldb_alogps_logp")),
            "logs_alogps": _safe_float(r.get("moldb_alogps_logs")),
            "solubility_alogps_g_per_l": _safe_float(r.get("moldb_alogps_solubility")),
            "pka_strongest_acidic": _safe_float(r.get("moldb_pka_strongest_acidic")),
            "pka_strongest_basic": _safe_float(r.get("moldb_pka_strongest_basic")),
            "physiological_charge": _safe_int(r.get("moldb_physiological_charge")),
            "hba": _safe_int(r.get("moldb_acceptor_count")),
            "hbd": _safe_int(r.get("moldb_donor_count")),
            "tpsa": _safe_float(r.get("moldb_polar_surface_area")),
            "rotatable_bonds": _safe_int(r.get("moldb_rotatable_bond_count")),
            "refractivity": _safe_float(r.get("moldb_refractivity")),
            "polarizability": _safe_float(r.get("moldb_polarizability")),
        }

        # Retention indices (flatten)
        retention_indices = []
        for ri in r.get("retention_indices") or []:
            retention_indices.append({
                "derivative_name": ri.get("derivative_name"),
                "retention_index": _safe_float(ri.get("retention_index")),
                "stationary_phase": ri.get("stationary_phase"),
                "pubmed_id": ri.get("pubmed_id"),
                "ref_text": ri.get("ref_text"),
            })

        # Evidence: keep ALL publications for each biospecimen/condition
        normal_evidence, normal_refs = _biospecimen_with_all_pubs(r.get("biospecimen_normal") or [], context="normal")
        abnormal_evidence, abnormal_refs = _biospecimen_with_all_pubs(r.get("biospecimen_abnormal") or [], context="abnormal")
        evidence = {
            "normal": normal_evidence,
            "abnormal": abnormal_evidence,
        }

        # Base references (general)
        general_refs = _flatten_refs(r.get("general_references") or [])

        # Retention index references (PMIDs) — tag as 'analytical'
        ri_refs = []
        for ri in r.get("retention_indices") or []:
            pmid = ri.get("pubmed_id")
            if pmid:
                ri_refs.append({
                    "pubmed_id": _norm_str(pmid) or None,
                    "title": None,
                    "source": ri.get("ref_text"),
                    "date": None,
                    "authors": None,
                    "context": "analytical",
                    "stationary_phase": ri.get("stationary_phase"),
                    "derivative_name": ri.get("derivative_name"),
                })

        # Synthesis reference as a pseudo-ref (no PMID)
        synth_ref_txt = r.get("synthesis_reference") or ""
        synth_refs = []
        if synth_ref_txt:
            synth_refs.append({
                "pubmed_id": None,
                "title": synth_ref_txt,
                "source": None,
                "date": None,
                "authors": None,
                "context": "synthesis",
            })

        # Combine + dedupe
        references_all = _dedupe_refs(general_refs + normal_refs + abnormal_refs + ri_refs + synth_refs)

        # References section (keep old shape + add references_all)
        references = {
            "synthesis_reference": synth_ref_txt,
            "general": [ {k: v for k, v in ref.items() if k != "context"} for ref in general_refs ],
            "all": references_all,  # NEW: flat, deduped list with context
        }

        notes = []
        if mono and rd_exact and abs(mono - rd_exact) > 0.005:
            notes.append(f"Monoisotopic mass ({mono} Da) differs from RDKit exact mass ({rd_exact} Da) by >5 mDa.")
        if not smiles and inchi:
            notes.append("No SMILES provided by HMDB; InChI present.")

        core = {
            "name": name,
            "hmdb_id": hmdb_id,
            "status": status,
            "formula": rd_formula or formula,
            "smiles": smiles,
            "inchi": inchi,
            "inchikey": inchikey,
            "mass_mono_da": rd_exact or mono,
            "mass_avg_da": avg,
            "cas": cas,
        }

        return {
            "core": core,
            "identifiers": identifiers,
            "taxonomy": taxonomy,
            "physchem": physchem,
            "retention_indices": retention_indices,
            "evidence": evidence,
            "references": references,   # includes "all"
            "notes": notes,
        }

    # ---------------------------
    # Rendering
    # ---------------------------
    def _format_payload(self, payload: Dict[str, Any], return_format: str) -> str:
        return_format = (return_format or "json").lower()
        js = json.dumps(payload, ensure_ascii=False)
        if return_format == "json":
            return js
        md = self._to_markdown(payload)
        if return_format == "markdown":
            return md
        if return_format == "both":
            return f"{js}\n\n---\n\n{md}"
        return js

    def _to_markdown(self, p: Dict[str, Any]) -> str:
        if p.get("error"):
            return f"**Error**: {p.get('message','Unknown error')}"
        lines: List[str] = ["### HMDB Metabolite Lookup"]

        prim = p.get("primary", {})
        lines.append("**Primary hit**")
        lines.append(f"- Name: `{prim.get('name','')}`")
        lines.append(f"- HMDB ID: `{prim.get('hmdb_id','')}`")
        if prim.get("status"):
            lines.append(f"- Status: `{prim['status']}`")
        lines.append(f"- Formula: `{prim.get('formula','')}`")
        if prim.get("mass_mono_da") is not None:
            lines.append(f"- Monoisotopic mass (Da): `{prim['mass_mono_da']}`")
        if prim.get("mass_avg_da") is not None:
            lines.append(f"- Average mass (Da): `{prim['mass_avg_da']}`")
        if prim.get("smiles"):
            lines.append(f"- SMILES: `{prim['smiles']}`")
        if prim.get("inchikey"):
            lines.append(f"- InChIKey: `{prim['inchikey']}`")
        if prim.get("cas"):
            lines.append(f"- CAS: `{prim['cas']}`")

        # Candidates
        cands = p.get("candidates") or []
        if len(cands) > 1:
            lines.append("\n**Candidates**")
            for i, c in enumerate(cands, 1):
                lines.append(f"{i}. `{c.get('name','')}` — HMDB:`{c.get('hmdb_id','')}` — {c.get('formula','')}`")

        # Synonyms
        syns = p.get("synonyms") or []
        if syns:
            lines.append(f"\n**Synonyms**: " + ", ".join(f"`{s}`" for s in syns[:30]))

        # Identifiers
        ids = p.get("identifiers") or {}
        lines.append("\n**External IDs**")
        for k in ["pubchem_cid", "chebi_id", "kegg_id", "drugbank_id", "chemspider_id", "metlin_id", "wikipedia", "markerdb_id", "foodb_id"]:
            v = ids.get(k)
            if v:
                lines.append(f"- {k}: `{v}`")

        # Taxonomy
        tax = p.get("taxonomy") or {}
        lines.append("\n**Chemical taxonomy**")
        for k in ["kingdom", "superclass", "class", "subclass", "direct_parent", "molecular_framework"]:
            v = tax.get(k)
            if v:
                lines.append(f"- {k.replace('_',' ').title()}: `{v}`")

        # Physchem
        phys = p.get("physchem") or {}
        lines.append("\n**Physicochemical (HMDB model)**")
        for k in ["logp_alogps", "logs_alogps", "solubility_alogps_g_per_l", "pka_strongest_acidic", "pka_strongest_basic",
                  "physiological_charge", "hba", "hbd", "tpsa", "rotatable_bonds", "refractivity", "polarizability"]:
            v = phys.get(k)
            if v is not None and v != "":
                lines.append(f"- {k.replace('_',' ').title()}: `{v}`")

        # Retention indices
        ris = p.get("retention_indices") or []
        if ris:
            lines.append("\n**GC Retention indices (selected)**")
            for ri in ris[:8]:
                idx = ri.get("retention_index")
                phase = ri.get("stationary_phase")
                deriv = ri.get("derivative_name")
                pmid = ri.get("pubmed_id")
                lines.append(f"- `{deriv}` | RI: `{idx}` | Phase: `{phase}`" + (f" | PMID: `{pmid}`" if pmid else ""))

        # Evidence
        ev = p.get("evidence") or {}
        for k in ["normal", "abnormal"]:
            items = ev.get(k) or []
            if items:
                lines.append(f"\n**Biospecimen evidence — {k}**")
                for it in items:
                    bios = it.get("biospecimen", "")
                    cond = it.get("condition", "")
                    pubs = it.get("publications") or []
                    if not pubs:
                        lines.append(f"- `{bios}` | `{cond}`")
                        continue
                    # Show all pubs per biospecimen/condition
                    for pub in pubs:
                        ttl = pub.get("title") or ""
                        pmid = pub.get("pubmed_id")
                        src  = pub.get("source")
                        bits = [f"`{bios}`", f"`{cond}`"]
                        if pmid: bits.append(f"PMID `{pmid}`")
                        if ttl:  bits.append(ttl)
                        if src:  bits.append(src)
                        lines.append("- " + " | ".join(bits))

        # References
        all_refs = (p.get("references") or {}).get("all") or []
        if all_refs:
            lines.append("\n**All references (flat, deduped)**")
            for ref in all_refs:
                bits = []
                if ref.get("context"):
                    bits.append(f"[{ref['context']}]")
                if ref.get("title"):
                    bits.append(ref["title"])
                if ref.get("pubmed_id"):
                    bits.append(f"PMID: {ref['pubmed_id']}")
                if ref.get("source"):
                    bits.append(ref["source"])
                if ref.get("biospecimen"):
                    bits.append(f"Specimen: {ref['biospecimen']}")
                if ref.get("condition"):
                    bits.append(f"Condition: {ref['condition']}")
                lines.append("- " + " | ".join(bits))

        # Notes
        notes = p.get("notes") or []
        if notes:
            lines.append("\n**Notes**")
            for n in notes:
                lines.append(f"- {n}")

        return "\n".join(lines)


# ---------------------------
# Helpers
# ---------------------------
def _safe_float(x) -> Optional[float]:
    if x is None or x == "":
        return None
    try:
        return float(x)
    except Exception:
        return None

def _safe_int(x) -> Optional[int]:
    if x is None or x == "":
        return None
    try:
        return int(float(x))
    except Exception:
        return None

def _flatten_refs(refs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out = []
    for r in refs:
        out.append({
            "pubmed_id": _norm_str(r.get("pubmed_id")) or None,
            "title": r.get("title"),
            "source": r.get("source") or r.get("publisher") or "",
            "date": r.get("date") or r.get("year"),
            "authors": r.get("authors"),
            "context": "general",
        })
    return out

def _biospecimen_with_all_pubs(bios: List[Dict[str, Any]], context: str) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Returns:
      evidence_items: list with ALL publications per biospecimen/condition
      refs: flattened references extracted from those publications (with context tags)
    """
    evidence_items, refs = [], []
    for b in bios:
        pubs = b.get("publications") or []
        pub_items = []
        for p in pubs:
            item = {
                "pubmed_id": _norm_str(p.get("pubmed_id")) or None,
                "title": p.get("title"),
                "source": p.get("source"),
                "date": p.get("date"),
                "authors": p.get("authors"),
            }
            pub_items.append(item)
            refs.append({
                **item,
                "context": context,  # "normal" or "abnormal"
                "biospecimen": b.get("biospecimen"),
                "condition": b.get("condition"),
            })
        evidence_items.append({
            "biospecimen": b.get("biospecimen"),
            "condition": b.get("condition"),
            "publications": pub_items,  # <-- ALL pubs retained here
        })
    return evidence_items, refs

def _norm_str(x) -> str:
    """Return a trimmed string for any primitive (int/float/str/None)."""
    if x is None:
        return ""
    try:
        s = str(x)
    except Exception:
        return ""
    return s.strip()

def _dedupe_refs(refs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Deduplicate by pubmed_id (preferred) or title; otherwise by (source,date,authors).
    Keys are tuples to avoid accidental collisions.
    """
    seen = set()
    out = []
    for r in refs:
        pmid = _norm_str(r.get("pubmed_id"))
        title = _norm_str(r.get("title"))
        if pmid:
            key = ("pmid", pmid)
        elif title:
            key = ("title", title)
        else:
            key = ("misc", _norm_str(r.get("source")), _norm_str(r.get("date")), _norm_str(r.get("authors")))
        if key in seen:
            continue
        seen.add(key)
        out.append(r)
    return out



In [14]:
tool = HMDBDatabaseTool()  # uses HMDB_API_KEY from env
print(
    tool._run(
        name="1-methylhistidine",
        return_format="both"  # "json" | "markdown" | "both"
    )
)

{"error": false, "primary": {"name": "1-Methylhistidine", "hmdb_id": "HMDB0000001", "status": "Detected and Quantified", "formula": "C7H11N3O2", "smiles": "CN1C=NC(C[C@H](N)C(O)=O)=C1", "inchi": "InChI=1S/C7H11N3O2/c1-10-3-5(9-4-10)2-6(8)7(11)12/h3-4,6H,2,8H2,1H3,(H,11,12)/t6-/m0/s1", "inchikey": "BRMWTNUJHUMWMS-LURJTMIESA-N", "mass_mono_da": 169.085127, "mass_avg_da": 169.1811, "cas": "332-80-9"}, "candidates": [{"name": "1-Methylhistidine", "hmdb_id": "HMDB0000001", "status": "Detected and Quantified", "formula": "C7H11N3O2", "smiles": "CN1C=NC(C[C@H](N)C(O)=O)=C1", "inchi": "InChI=1S/C7H11N3O2/c1-10-3-5(9-4-10)2-6(8)7(11)12/h3-4,6H,2,8H2,1H3,(H,11,12)/t6-/m0/s1", "inchikey": "BRMWTNUJHUMWMS-LURJTMIESA-N", "mass_mono_da": 169.085127, "mass_avg_da": 169.1811, "cas": "332-80-9"}, {"name": "N-Acetyl-1-methylhistidine", "hmdb_id": "HMDB0240340", "status": "Detected but not Quantified", "formula": "C9H13N3O3", "smiles": "CN1C=NC(C[C@H](NC(C)=O)C(O)=O)=C1", "inchi": "InChI=1S/C9H13N3O3/c1-